# P3: Dockerized ML API

**Module 3 — Week 12: MLOps & Production Systems**

---

## Problem Statement

Build a production-grade ML API that serves a churn prediction model via FastAPI, with
proper input validation, batch inference, health checks, metrics collection, and Docker
containerization. The API must be robust enough for production deployment.

## Success Metrics

| Metric | Target |
|--------|--------|
| p95 latency | < 100 ms |
| Error rate | < 1% |
| All endpoints tested | /health, /v1/predict, /v1/predict/batch, /model/info, /metrics |
| Input validation | Rejects malformed payloads with 422 |
| Containerisation | Dockerfile builds and runs cleanly |

## Approach

1. Train a GradientBoosting model on synthetic churn data
2. Build a FastAPI application with all endpoints in-notebook (mirrors `src/`)
3. Test every endpoint with `TestClient` (no running server needed)
4. Validate error handling and edge cases
5. Walk through Docker containerization
6. Load-test the API and measure latency percentiles
7. Review a production-readiness checklist

In [ ]:
# Imports
import sys, os, time, json, pickle, logging
from io import BytesIO
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"Python {sys.version}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

---
## 1. Setup — Train the Churn Model

We generate synthetic customer data and train a GradientBoosting classifier entirely
in-notebook so the rest of the notebook is self-contained (no `model.pkl` on disk required).

In [ ]:
# ---------------------------------------------------------------------------
# Feature names (must match customer_to_array ordering)
# ---------------------------------------------------------------------------
FEATURE_NAMES = [
    "tenure_months", "monthly_charges", "total_charges",
    "contract_One year", "contract_Two year",
    "internet_Fiber optic", "internet_No",
    "online_security", "tech_support", "senior_citizen", "partner",
]


def generate_training_data(n: int = 5000, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic churn data for model training."""
    np.random.seed(seed)

    tenure = np.random.exponential(32, n).clip(1, 72).astype(int)
    contract = np.random.choice(
        ["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.20]
    )
    internet = np.random.choice(
        ["DSL", "Fiber optic", "No"], n, p=[0.34, 0.44, 0.22]
    )
    monthly = np.round(
        20 + 25 * (internet == "DSL") + 40 * (internet == "Fiber optic")
        + np.random.normal(0, 8, n), 2,
    ).clip(18, 120)
    total = np.round(monthly * tenure * (1 + np.random.normal(0, 0.05, n)), 2)
    online_sec = np.where(internet == "No", 0, np.random.binomial(1, 0.35, n))
    tech_sup = np.where(internet == "No", 0, np.random.binomial(1, 0.30, n))
    senior = np.random.binomial(1, 0.16, n)
    partner = np.random.binomial(1, 0.48, n)

    logit = (
        -1.5 - 0.04 * tenure + 1.2 * (contract == "Month-to-month")
        - 0.8 * (contract == "Two year") + 0.02 * monthly
        + 0.5 * (internet == "Fiber optic") - 0.5 * online_sec
        - 0.4 * tech_sup + 0.3 * senior - 0.3 * partner
        + np.random.normal(0, 0.5, n)
    )
    churn = (np.random.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    return pd.DataFrame({
        "tenure_months": tenure,
        "monthly_charges": monthly,
        "total_charges": total,
        "contract": contract,
        "internet_service": internet,
        "online_security": online_sec,
        "tech_support": tech_sup,
        "senior_citizen": senior,
        "partner": partner,
        "churned": churn,
    })


def prepare_features(df: pd.DataFrame) -> np.ndarray:
    """Convert DataFrame to feature array matching FEATURE_NAMES."""
    return np.column_stack([
        df["tenure_months"].values,
        df["monthly_charges"].values,
        df["total_charges"].values,
        (df["contract"] == "One year").astype(int).values,
        (df["contract"] == "Two year").astype(int).values,
        (df["internet_service"] == "Fiber optic").astype(int).values,
        (df["internet_service"] == "No").astype(int).values,
        df["online_security"].values,
        df["tech_support"].values,
        df["senior_citizen"].values,
        df["partner"].values,
    ])


# Generate data
df = generate_training_data(n=5000, seed=42)
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Churn rate: {df['churned'].mean():.1%}")
df.head(3)

In [ ]:
# ---------------------------------------------------------------------------
# Train GradientBoosting model
# ---------------------------------------------------------------------------
X = prepare_features(df)
y = df["churned"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=== Churn Model Training Results ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))

In [ ]:
# ---------------------------------------------------------------------------
# Package into an in-memory artifact (same structure as src/train_model.py)
# ---------------------------------------------------------------------------
model_artifact = {
    "model": model,
    "feature_names": FEATURE_NAMES,
    "version": "1.0.0",
    "metrics": {
        "accuracy": float(acc),
        "f1": float(f1),
        "auc": float(auc),
    },
}

# Quick serialization round-trip to confirm the artifact is pickle-safe
buf = BytesIO()
pickle.dump(model_artifact, buf)
buf.seek(0)
_loaded = pickle.load(buf)
assert _loaded["version"] == "1.0.0"
print(f"Model artifact OK  |  pickle size: {buf.tell() / 1024:.1f} KB")
print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

---
## 2. Build the FastAPI Application

Below we recreate the full API from `src/main.py` and `src/schemas.py` as self-contained
notebook code. This lets us use FastAPI's `TestClient` without starting a server process.

In [ ]:
# ---------------------------------------------------------------------------
# Pydantic schemas (mirrors src/schemas.py)
# ---------------------------------------------------------------------------
from pydantic import BaseModel, Field
from typing import List, Optional


class CustomerFeatures(BaseModel):
    """Input features for a single customer churn prediction."""
    tenure_months: int = Field(ge=0, le=120, description="Months as customer")
    monthly_charges: float = Field(ge=0, le=500, description="Monthly bill amount")
    total_charges: float = Field(ge=0, description="Total charges to date")
    contract: str = Field(
        description="Contract type",
        pattern="^(Month-to-month|One year|Two year)$",
    )
    internet_service: str = Field(
        description="Internet type",
        pattern="^(DSL|Fiber optic|No)$",
    )
    online_security: int = Field(ge=0, le=1, description="Has online security")
    tech_support: int = Field(ge=0, le=1, description="Has tech support")
    senior_citizen: int = Field(ge=0, le=1, default=0)
    partner: int = Field(ge=0, le=1, default=0)

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "tenure_months": 12,
                    "monthly_charges": 79.99,
                    "total_charges": 959.88,
                    "contract": "Month-to-month",
                    "internet_service": "Fiber optic",
                    "online_security": 0,
                    "tech_support": 0,
                    "senior_citizen": 0,
                    "partner": 1,
                }
            ]
        }
    }


class PredictionResponse(BaseModel):
    """Response for a single churn prediction."""
    churn_probability: float
    churn_prediction: bool
    model_version: str = "1.0.0"


class BatchPredictionRequest(BaseModel):
    """Request body for batch predictions."""
    customers: List[CustomerFeatures]


class BatchPredictionResponse(BaseModel):
    """Response for batch predictions."""
    predictions: List[PredictionResponse]
    count: int


class HealthResponse(BaseModel):
    """Health-check response."""
    status: str
    model_loaded: bool
    model_version: str
    uptime_seconds: float


print("Schemas defined: CustomerFeatures, PredictionResponse, "
      "BatchPredictionRequest, BatchPredictionResponse, HealthResponse")

In [ ]:
# ---------------------------------------------------------------------------
# FastAPI app (mirrors src/main.py, adapted for in-notebook usage)
# ---------------------------------------------------------------------------
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

# Globals
app_start_time = time.time()
request_metrics: dict = defaultdict(list)


def customer_to_array(c: CustomerFeatures) -> np.ndarray:
    """Convert a CustomerFeatures Pydantic model to a numpy array."""
    return np.array([[
        c.tenure_months,
        c.monthly_charges,
        c.total_charges,
        int(c.contract == "One year"),
        int(c.contract == "Two year"),
        int(c.internet_service == "Fiber optic"),
        int(c.internet_service == "No"),
        c.online_security,
        c.tech_support,
        c.senior_citizen,
        c.partner,
    ]])


# --- App ---
app = FastAPI(
    title="Churn Prediction API",
    description="Production ML service for customer churn prediction.",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# --- Middleware ---
@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    t0 = time.time()
    response = await call_next(request)
    elapsed_ms = (time.time() - t0) * 1000
    response.headers["X-Response-Time-Ms"] = f"{elapsed_ms:.2f}"
    request_metrics["latencies"].append(elapsed_ms)
    request_metrics["total_requests"].append(1)
    return response


# --- Endpoints ---
@app.get("/health", response_model=HealthResponse)
def health():
    return HealthResponse(
        status="healthy",
        model_loaded=model_artifact is not None,
        model_version=model_artifact["version"] if model_artifact else "N/A",
        uptime_seconds=round(time.time() - app_start_time, 1),
    )


@app.post("/v1/predict", response_model=PredictionResponse)
def predict(customer: CustomerFeatures):
    if model_artifact is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    X = customer_to_array(customer)
    mdl = model_artifact["model"]
    proba = float(mdl.predict_proba(X)[0, 1])
    return PredictionResponse(
        churn_probability=round(proba, 4),
        churn_prediction=proba >= 0.5,
        model_version=model_artifact["version"],
    )


@app.post("/v1/predict/batch", response_model=BatchPredictionResponse)
def predict_batch(request: BatchPredictionRequest):
    if model_artifact is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    if len(request.customers) == 0:
        raise HTTPException(status_code=400, detail="Empty batch")

    X = np.vstack([customer_to_array(c) for c in request.customers])
    mdl = model_artifact["model"]
    probas = mdl.predict_proba(X)[:, 1]

    predictions = [
        PredictionResponse(
            churn_probability=round(float(p), 4),
            churn_prediction=p >= 0.5,
            model_version=model_artifact["version"],
        )
        for p in probas
    ]
    return BatchPredictionResponse(predictions=predictions, count=len(predictions))


@app.get("/model/info")
def model_info():
    if model_artifact is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    return {
        "version": model_artifact["version"],
        "features": model_artifact["feature_names"],
        "metrics": model_artifact["metrics"],
    }


@app.get("/metrics")
def metrics_endpoint():
    latencies = request_metrics["latencies"]
    total = len(request_metrics["total_requests"])
    return {
        "total_requests": total,
        "avg_latency_ms": round(np.mean(latencies), 2) if latencies else 0,
        "p50_latency_ms": round(np.percentile(latencies, 50), 2) if latencies else 0,
        "p95_latency_ms": round(np.percentile(latencies, 95), 2) if latencies else 0,
        "p99_latency_ms": round(np.percentile(latencies, 99), 2) if latencies else 0,
    }


# Create the test client (no server process needed)
client = TestClient(app)
print("FastAPI app created with endpoints:")
for route in app.routes:
    if hasattr(route, 'methods'):
        for method in route.methods:
            print(f"  {method:6s} {route.path}")

---
## 3. Test with TestClient

FastAPI ships with `TestClient` (powered by `httpx` / Starlette), which lets us
exercise every endpoint synchronously without starting a live server.

In [ ]:
# ---------------------------------------------------------------------------
# Helper: a valid customer payload reused across tests
# ---------------------------------------------------------------------------
VALID_CUSTOMER = {
    "tenure_months": 12,
    "monthly_charges": 79.99,
    "total_charges": 959.88,
    "contract": "Month-to-month",
    "internet_service": "Fiber optic",
    "online_security": 0,
    "tech_support": 0,
    "senior_citizen": 0,
    "partner": 1,
}

print("Valid customer payload:")
print(json.dumps(VALID_CUSTOMER, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# TODO: test_health -- GET /health, assert 200, check response fields
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Send a GET request to /health using `client`.
# Assert status code is 200.
# Parse the JSON response and verify:
#   - status == "healthy"
#   - model_loaded == True
#   - model_version is present
#   - uptime_seconds >= 0

# --- Solution (uncomment to run) ---
# resp = client.get("/health")
# assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
# data = resp.json()
# assert data["status"] == "healthy"
# assert data["model_loaded"] is True
# assert "model_version" in data
# assert data["uptime_seconds"] >= 0
# print("test_health PASSED")
# print(json.dumps(data, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# TODO: test_predict_valid -- POST /v1/predict with a valid customer
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Send a POST to /v1/predict with VALID_CUSTOMER as JSON.
# Assert status 200.
# Verify response contains:
#   - churn_probability (float between 0 and 1)
#   - churn_prediction (bool)
#   - model_version (str)

# --- Solution (uncomment to run) ---
# resp = client.post("/v1/predict", json=VALID_CUSTOMER)
# assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
# data = resp.json()
# assert 0.0 <= data["churn_probability"] <= 1.0
# assert isinstance(data["churn_prediction"], bool)
# assert isinstance(data["model_version"], str)
# print("test_predict_valid PASSED")
# print(f"  Churn probability: {data['churn_probability']:.4f}")
# print(f"  Churn prediction:  {data['churn_prediction']}")

In [ ]:
# ---------------------------------------------------------------------------
# TODO: test_predict_invalid -- POST /v1/predict with invalid input
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Send a POST to /v1/predict with tenure_months = -5 (violates ge=0).
# Assert status code is 422 (Unprocessable Entity).
# Verify the error response contains "detail" with validation info.

# --- Solution (uncomment to run) ---
# bad_customer = VALID_CUSTOMER.copy()
# bad_customer["tenure_months"] = -5
# resp = client.post("/v1/predict", json=bad_customer)
# assert resp.status_code == 422, f"Expected 422, got {resp.status_code}"
# errors = resp.json()["detail"]
# assert len(errors) > 0, "Expected validation errors"
# print("test_predict_invalid PASSED")
# print(f"  Validation errors: {len(errors)}")
# for err in errors:
#     print(f"    - {err['loc']}: {err['msg']}")

In [ ]:
# ---------------------------------------------------------------------------
# TODO: test_predict_batch -- POST /v1/predict/batch with 5 customers
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Create a batch of 5 customers with varying feature values.
# Send POST to /v1/predict/batch with {"customers": [...]}.
# Assert status 200.
# Verify response has "predictions" list of length 5 and "count" == 5.

# --- Solution (uncomment to run) ---
# batch_customers = []
# contracts = ["Month-to-month", "One year", "Two year", "Month-to-month", "One year"]
# internets = ["Fiber optic", "DSL", "No", "DSL", "Fiber optic"]
# for i in range(5):
#     batch_customers.append({
#         "tenure_months": 6 + i * 12,
#         "monthly_charges": 40.0 + i * 15,
#         "total_charges": (40.0 + i * 15) * (6 + i * 12),
#         "contract": contracts[i],
#         "internet_service": internets[i],
#         "online_security": i % 2,
#         "tech_support": (i + 1) % 2,
#         "senior_citizen": 0,
#         "partner": 1,
#     })
#
# resp = client.post("/v1/predict/batch", json={"customers": batch_customers})
# assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
# data = resp.json()
# assert data["count"] == 5
# assert len(data["predictions"]) == 5
# print("test_predict_batch PASSED")
# print(f"  Batch count: {data['count']}")
# for i, pred in enumerate(data["predictions"]):
#     print(f"    Customer {i}: p(churn)={pred['churn_probability']:.4f}  "
#           f"churn={pred['churn_prediction']}")

---
## 4. Error Handling & Edge Cases

Robust APIs must handle all sorts of malformed input gracefully. We test:
- Missing required fields
- Wrong data types
- Empty batch requests
- Invalid enum/pattern values

In [ ]:
# ---------------------------------------------------------------------------
# TODO: test edge cases
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Test 1: Missing required fields -- send {} to /v1/predict, expect 422
# Test 2: Wrong types -- send tenure_months="abc", expect 422
# Test 3: Empty batch -- send {"customers": []} to /v1/predict/batch, expect 400
# Test 4: Invalid contract value -- send contract="Weekly", expect 422
# Test 5: Out-of-range monthly_charges -- send monthly_charges=999, expect 422
#
# For each test, assert the expected status code and print a PASS message.

# --- Solution (uncomment to run) ---
# test_cases = [
#     {
#         "name": "Missing required fields",
#         "payload": {},
#         "expected_status": 422,
#         "endpoint": "/v1/predict",
#     },
#     {
#         "name": "Wrong type for tenure_months",
#         "payload": {**VALID_CUSTOMER, "tenure_months": "abc"},
#         "expected_status": 422,
#         "endpoint": "/v1/predict",
#     },
#     {
#         "name": "Empty batch",
#         "payload": {"customers": []},
#         "expected_status": 400,
#         "endpoint": "/v1/predict/batch",
#     },
#     {
#         "name": "Invalid contract value",
#         "payload": {**VALID_CUSTOMER, "contract": "Weekly"},
#         "expected_status": 422,
#         "endpoint": "/v1/predict",
#     },
#     {
#         "name": "monthly_charges out of range (>500)",
#         "payload": {**VALID_CUSTOMER, "monthly_charges": 999.0},
#         "expected_status": 422,
#         "endpoint": "/v1/predict",
#     },
# ]
#
# print("Edge-case error handling tests:")
# for tc in test_cases:
#     resp = client.post(tc["endpoint"], json=tc["payload"])
#     status_ok = resp.status_code == tc["expected_status"]
#     tag = "PASSED" if status_ok else "FAILED"
#     print(f"  [{tag}] {tc['name']}: "
#           f"expected {tc['expected_status']}, got {resp.status_code}")
#     assert status_ok, f"Test failed: {tc['name']}"
#
# print("\nAll edge-case tests PASSED")

---
## 5. Containerize with Docker

The project includes a multi-stage `Dockerfile` and `docker-compose.yml` for easy
deployment. Below we review the Dockerfile and the commands to build/run.

In [ ]:
# ---------------------------------------------------------------------------
# Print the Dockerfile for reference
# ---------------------------------------------------------------------------
dockerfile_content = """\
# Stage 1: Install dependencies
FROM python:3.11-slim AS builder

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/app/deps -r requirements.txt

# Stage 2: Runtime
FROM python:3.11-slim

WORKDIR /app

# Copy installed packages
COPY --from=builder /app/deps /usr/local/lib/python3.11/site-packages/

# Copy application code
COPY src/ ./src/
COPY model.pkl .

# Non-root user for security
RUN useradd -m -r appuser && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1

CMD ["python", "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print("=== Dockerfile ===")
print(dockerfile_content)

In [ ]:
# ---------------------------------------------------------------------------
# Print docker-compose.yml for reference
# ---------------------------------------------------------------------------
compose_content = """\
version: "3.8"

services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model.pkl
      - LOG_LEVEL=info
    volumes:
      - ./model.pkl:/app/model.pkl:ro
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
"""

print("=== docker-compose.yml ===")
print(compose_content)

### Docker Build & Run Commands

```bash
# 1. Train the model (creates model.pkl)
python -m src.train_model

# 2. Build the Docker image
docker build -t churn-api:latest .

# 3. Run the container
docker run -d -p 8000:8000 --name churn-api churn-api:latest

# 4. Test the health endpoint
curl http://localhost:8000/health

# 5. Test a prediction
curl -X POST http://localhost:8000/v1/predict \
  -H "Content-Type: application/json" \
  -d '{"tenure_months":12,"monthly_charges":79.99,"total_charges":959.88,\
       "contract":"Month-to-month","internet_service":"Fiber optic",\
       "online_security":0,"tech_support":0}'

# Or use docker compose:
docker compose up -d
docker compose logs -f
docker compose down
```

**Key Dockerfile features:**
- Multi-stage build to minimize image size
- Non-root user (`appuser`) for security
- Built-in `HEALTHCHECK` for orchestration readiness
- `EXPOSE 8000` documents the listening port

---
## 6. Integration Testing

Comprehensive integration tests exercise the full request/response cycle across
all endpoints, verifying status codes, response shapes, and data correctness.

In [ ]:
# ---------------------------------------------------------------------------
# TODO: Write comprehensive integration tests covering all endpoints
# ---------------------------------------------------------------------------

# YOUR CODE HERE
#
# Write a function `run_integration_tests(client)` that tests:
#   1. GET  /health           -> 200, correct fields
#   2. POST /v1/predict       -> 200, probability in [0,1]
#   3. POST /v1/predict       -> 422 for invalid input
#   4. POST /v1/predict/batch -> 200, correct count
#   5. POST /v1/predict/batch -> 400 for empty batch
#   6. GET  /model/info       -> 200, has version/features/metrics
#   7. GET  /metrics          -> 200, has total_requests
#   8. Verify response time header (X-Response-Time-Ms) is present
#
# Print a summary of passed/failed tests.

# --- Solution (uncomment to run) ---
# def run_integration_tests(client):
#     """Run all integration tests and return results."""
#     results = []
#
#     # Test 1: Health endpoint
#     resp = client.get("/health")
#     ok = resp.status_code == 200 and resp.json()["status"] == "healthy"
#     results.append(("GET /health", ok))
#
#     # Test 2: Valid prediction
#     resp = client.post("/v1/predict", json=VALID_CUSTOMER)
#     data = resp.json()
#     ok = (resp.status_code == 200
#           and 0 <= data["churn_probability"] <= 1
#           and isinstance(data["churn_prediction"], bool))
#     results.append(("POST /v1/predict (valid)", ok))
#
#     # Test 3: Invalid prediction
#     bad = {**VALID_CUSTOMER, "tenure_months": -1}
#     resp = client.post("/v1/predict", json=bad)
#     ok = resp.status_code == 422
#     results.append(("POST /v1/predict (invalid)", ok))
#
#     # Test 4: Batch prediction
#     batch = {"customers": [VALID_CUSTOMER, VALID_CUSTOMER, VALID_CUSTOMER]}
#     resp = client.post("/v1/predict/batch", json=batch)
#     data = resp.json()
#     ok = resp.status_code == 200 and data["count"] == 3
#     results.append(("POST /v1/predict/batch (3 customers)", ok))
#
#     # Test 5: Empty batch
#     resp = client.post("/v1/predict/batch", json={"customers": []})
#     ok = resp.status_code == 400
#     results.append(("POST /v1/predict/batch (empty)", ok))
#
#     # Test 6: Model info
#     resp = client.get("/model/info")
#     data = resp.json()
#     ok = (resp.status_code == 200
#           and "version" in data
#           and "features" in data
#           and "metrics" in data)
#     results.append(("GET /model/info", ok))
#
#     # Test 7: Metrics endpoint
#     resp = client.get("/metrics")
#     data = resp.json()
#     ok = resp.status_code == 200 and "total_requests" in data
#     results.append(("GET /metrics", ok))
#
#     # Test 8: Response time header
#     resp = client.get("/health")
#     ok = "x-response-time-ms" in resp.headers
#     results.append(("X-Response-Time-Ms header present", ok))
#
#     # Summary
#     passed = sum(1 for _, ok in results if ok)
#     total = len(results)
#     print(f"\n{'='*55}")
#     print(f"Integration Tests: {passed}/{total} passed")
#     print(f"{'='*55}")
#     for name, ok in results:
#         tag = "PASS" if ok else "FAIL"
#         print(f"  [{tag}] {name}")
#     print()
#     return results
#
# integration_results = run_integration_tests(client)

---
## 7. Load Testing

We simulate concurrent API traffic using `ThreadPoolExecutor` to measure latency
under load and verify our p95 target (< 100 ms).

In [ ]:
# ---------------------------------------------------------------------------
# TODO: Implement load_test and run it
# ---------------------------------------------------------------------------

# YOUR CODE HERE
#
# def load_test(client, n_requests=200, n_workers=10):
#     """Send concurrent requests using ThreadPoolExecutor.
#
#     Returns a dict with:
#       - latencies: list of floats (ms)
#       - errors: count of non-200 responses
#       - total: total requests sent
#       - elapsed: total wall-clock time (seconds)
#     """
#     # For each request:
#     #   1. Record start time
#     #   2. POST /v1/predict with VALID_CUSTOMER
#     #   3. Record end time and status code
#     #   4. Collect latency = (end - start) * 1000
#     pass
#
# Run the load test and print summary statistics:
#   - Total requests, errors, throughput (req/s)
#   - p50, p95, p99, max latency

# --- Solution (uncomment to run) ---
# def load_test(client, n_requests=200, n_workers=10):
#     """Send concurrent prediction requests and measure latency."""
#     latencies = []
#     errors = 0
#     order = []  # (request_index, latency_ms)
#
#     def send_request(idx):
#         t0 = time.time()
#         resp = client.post("/v1/predict", json=VALID_CUSTOMER)
#         elapsed_ms = (time.time() - t0) * 1000
#         return idx, elapsed_ms, resp.status_code
#
#     wall_start = time.time()
#     with ThreadPoolExecutor(max_workers=n_workers) as executor:
#         futures = {executor.submit(send_request, i): i
#                    for i in range(n_requests)}
#         for future in as_completed(futures):
#             idx, lat, status = future.result()
#             latencies.append(lat)
#             order.append((idx, lat))
#             if status != 200:
#                 errors += 1
#     wall_elapsed = time.time() - wall_start
#
#     # Sort by request index to get time-ordered latencies
#     order.sort(key=lambda x: x[0])
#     ordered_latencies = [lat for _, lat in order]
#
#     return {
#         "latencies": latencies,
#         "ordered_latencies": ordered_latencies,
#         "errors": errors,
#         "total": n_requests,
#         "elapsed": wall_elapsed,
#     }
#
# print("Running load test: 200 requests, 10 concurrent workers...")
# load_results = load_test(client, n_requests=200, n_workers=10)
#
# lats = np.array(load_results["latencies"])
# throughput = load_results["total"] / load_results["elapsed"]
# error_rate = load_results["errors"] / load_results["total"] * 100
#
# print(f"\n{'='*50}")
# print(f"Load Test Results")
# print(f"{'='*50}")
# print(f"  Total requests : {load_results['total']}")
# print(f"  Errors         : {load_results['errors']} ({error_rate:.1f}%)")
# print(f"  Wall time      : {load_results['elapsed']:.2f}s")
# print(f"  Throughput     : {throughput:.1f} req/s")
# print(f"  ---")
# print(f"  p50 latency    : {np.percentile(lats, 50):.2f} ms")
# print(f"  p95 latency    : {np.percentile(lats, 95):.2f} ms")
# print(f"  p99 latency    : {np.percentile(lats, 99):.2f} ms")
# print(f"  max latency    : {np.max(lats):.2f} ms")
# print(f"  mean latency   : {np.mean(lats):.2f} ms")
#
# p95 = np.percentile(lats, 95)
# if p95 < 100:
#     print(f"\n  p95 < 100ms target: PASSED ({p95:.2f} ms)")
# else:
#     print(f"\n  p95 < 100ms target: FAILED ({p95:.2f} ms)")
#
# if error_rate < 1.0:
#     print(f"  Error rate < 1% target: PASSED ({error_rate:.1f}%)")
# else:
#     print(f"  Error rate < 1% target: FAILED ({error_rate:.1f}%)")

In [ ]:
# ---------------------------------------------------------------------------
# TODO: Plot latency distribution and latency over time
# ---------------------------------------------------------------------------

# YOUR CODE HERE
# Create a 1x2 figure:
#   Left:  Histogram of latencies with vertical lines for p50, p95, p99
#   Right: Line plot of ordered latencies (latency over request number)

# --- Solution (uncomment to run) ---
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# # Left: Histogram
# ax = axes[0]
# ax.hist(lats, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
# for pct, color, label in [(50, "green", "p50"), (95, "orange", "p95"), (99, "red", "p99")]:
#     val = np.percentile(lats, pct)
#     ax.axvline(val, color=color, linestyle="--", linewidth=1.5, label=f"{label}={val:.1f}ms")
# ax.set_xlabel("Latency (ms)")
# ax.set_ylabel("Count")
# ax.set_title("Latency Distribution")
# ax.legend()
#
# # Right: Latency over time
# ax = axes[1]
# ordered = load_results["ordered_latencies"]
# ax.plot(range(len(ordered)), ordered, color="steelblue", alpha=0.6, linewidth=0.8)
# ax.axhline(np.percentile(lats, 95), color="orange", linestyle="--",
#            linewidth=1.5, label=f"p95={np.percentile(lats, 95):.1f}ms")
# ax.axhline(100, color="red", linestyle=":", linewidth=1.5, label="Target (100ms)")
# ax.set_xlabel("Request #")
# ax.set_ylabel("Latency (ms)")
# ax.set_title("Latency Over Time")
# ax.legend()
#
# plt.tight_layout()
# plt.savefig("outputs/load_test_latency.png", dpi=150, bbox_inches="tight")
# plt.show()
# print("Saved: outputs/load_test_latency.png")

---
## 8. Production Readiness Checklist

| Category | Item | Status |
|----------|------|--------|
| **Observability** | Structured logging | Done |
| | Request timing middleware | Done |
| | /metrics endpoint | Done |
| | Monitoring / alerting (Prometheus, Grafana) | TODO |
| **Security** | CORS middleware | Done |
| | Non-root Docker user | Done |
| | Input validation (Pydantic) | Done |
| | Rate limiting | TODO |
| | Authentication / API keys | TODO |
| **Reliability** | Health check endpoint | Done |
| | Docker HEALTHCHECK | Done |
| | Graceful shutdown | TODO |
| | Circuit breakers | TODO |
| | Retry logic (client-side) | TODO |
| **Deployment** | Dockerfile (multi-stage) | Done |
| | docker-compose.yml | Done |
| | CI/CD pipeline | TODO |
| | Blue/green or canary deploy | TODO |
| **Documentation** | Auto-generated OpenAPI (Swagger) | Done |
| | API versioning (/v1/predict) | Done |
| | Pydantic schema examples | Done |

### Next Steps

1. **Rate limiting** -- Add `slowapi` or similar middleware to prevent abuse.
2. **Graceful shutdown** -- Handle SIGTERM to drain in-flight requests before exiting.
3. **Prometheus metrics** -- Replace the simple /metrics with `prometheus_client` for
   standard scraping by monitoring stacks.
4. **CI/CD** -- Add GitHub Actions workflow: lint, test, build image, push to registry.
5. **Model registry** -- Track model versions with MLflow or similar.

---
## 9. What I Built / What I Learned

In [ ]:
# ---------------------------------------------------------------------------
# Summary of what was built
# ---------------------------------------------------------------------------
print("="*60)
print("PROJECT SUMMARY: Dockerized ML API")
print("="*60)

print("\n--- Model Metrics ---")
for metric, value in model_artifact["metrics"].items():
    print(f"  {metric:10s}: {value:.4f}")

print(f"\n--- API Endpoints ---")
endpoints = [
    ("GET",  "/health",          "Health check with model status and uptime"),
    ("POST", "/v1/predict",      "Single customer churn prediction"),
    ("POST", "/v1/predict/batch","Batch predictions for multiple customers"),
    ("GET",  "/model/info",      "Model version, features, and training metrics"),
    ("GET",  "/metrics",         "Request count and latency percentiles"),
]
for method, path, desc in endpoints:
    print(f"  {method:5s} {path:22s}  {desc}")

print(f"\n--- Production Features ---")
features = [
    "Pydantic input validation with regex patterns and range constraints",
    "CORS middleware for cross-origin requests",
    "Request timing middleware (X-Response-Time-Ms header)",
    "Multi-stage Dockerfile with non-root user",
    "Docker HEALTHCHECK for orchestration readiness",
    "Auto-generated OpenAPI/Swagger documentation",
    "Batch inference endpoint for throughput",
    "Structured logging",
]
for feat in features:
    print(f"  - {feat}")

print(f"\n--- Success Metrics ---")
print(f"  p95 latency < 100ms : (run load test above to verify)")
print(f"  Error rate < 1%     : (run load test above to verify)")
print(f"  All endpoints tested: (run integration tests above to verify)")

In [ ]:
# ---------------------------------------------------------------------------
# TODO: What I Learned -- fill in your key takeaways
# ---------------------------------------------------------------------------

key_learnings = [
    # TODO: Replace these placeholders with your actual learnings
    "1. ...",
    "2. ...",
    "3. ...",
    "4. ...",
    "5. ...",
]

# Example learnings (uncomment and customize):
# key_learnings = [
#     "1. FastAPI + Pydantic gives you input validation and OpenAPI docs for free.",
#     "2. TestClient lets you test the full request/response cycle without a running server.",
#     "3. Multi-stage Docker builds significantly reduce image size.",
#     "4. Running as non-root in Docker is a simple but important security practice.",
#     "5. Latency percentiles (p95, p99) are more informative than mean latency.",
# ]

print("\nKey Learnings:")
for learning in key_learnings:
    print(f"  {learning}")

print("\n" + "="*60)
print("End of P3: Dockerized ML API")
print("="*60)